In [1]:
# Import necessary libraries
import json
import re
import logging
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from collections import Counter, defaultdict
import pandas as pd
from datetime import datetime
from pathlib import Path
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [60]:
# Configuration settings
CONFIG = {
    "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
    "use_quantization": True,
    "device": device,
    "max_new_tokens": 500,
    "temperature": 0.1,
    "input_file": "output_entities2.json",  # Update this path
    "output_dir": "lulc_extraction_output",
    "max_sentences": 20,  # Set to limit processing (e.g., 50)
}

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Define valid LULC relations
VALID_RELATIONS = [
    "CHANGE_TO",
    "INCREASES_BY",
    "DECREASES_BY",
    "CAUSES",
    "LOCATED_IN",
    "OCCURS_DURING",
    "MEASURES",
    "AFFECTS",
    "FROM_TO",
    "ENABLES"
]

print("Configuration loaded successfully")

Configuration loaded successfully


In [61]:
import pandas as pd

# Path to your CSV file
csv_file_path = "output_entities2.json"  # Update this path

# Read the CSV file
try:
    df = pd.read_json("output_entities2.json")
    print("CSV File Content:")
    print(df.head())  # Display the first few rows

    # Check if the CSV contains sentences and entities
    if 'sentence' in df.columns and 'entities' in df.columns:
        print("CSV contains sentences and entities.")
    else:
        print("CSV does not contain the required columns (sentence, entities).")
        exit(1)

    # Test with a sample sentence from the CSV
    sample_sentence = df.iloc[0]['sentence']
    sample_entities = df.iloc[0]['entities']
    sample_relations = df.iloc[0]['relations']

    print("\nSample Sentence:")
    print(sample_sentence)
    print("\nSample Entities:")
    print(sample_entities)
    print("\nSample Relations:")
    print(sample_relations)

    # Generate the prompt for the sample sentence
    prompt = build_lulc_extraction_prompt(sample_sentence)
    print("\nGenerated Prompt:")
    print(prompt)

    # Simulate the LLM response (for testing purposes)
    simulated_response = f"""
ENTITIES:
{sample_entities}

RELATIONS:
{sample_relations}
"""

    # Parse the simulated response
    entities, relations = parse_extraction_response(simulated_response)
    print("\nExtracted Entities:")
    for entity in entities:
        print(entity)

    print("\nExtracted Relations:")
    for relation in relations:
        print(relation)

except Exception as e:
    print(f"Error: {e}")

CSV File Content:
                                            sentence  \
0  text': 'After the droughts in the 1970s and 19...   
1  text': 'The forests of West and Central Africa...   
2  ).',, 'p': 'ref':, ' text': 'From 1996 to 2017...   
3  in this region has been dominated over the pas...   
4  land was decreased nearly by half in 2002 comp...   

                                            entities  
0  [{'text': 'the 1970s and 1980s', 'label': 'DAT...  
1  [{'text': 'forests', 'label': 'LULC', 'start':...  
2  [{'text': '1996 to 2017', 'label': 'DATE', 'st...  
3  [{'text': 'the past three decades', 'label': '...  
4  [{'text': 'decreased', 'label': 'CHANGE', 'sta...  
CSV contains sentences and entities.
Error: 'relations'


In [62]:
import json
import logging
import traceback

logger = logging.getLogger(__name__) # Initialize logger

def load_data_from_custom_format(file_path):
    """
    Loads data from a custom JSON format where each item has 'sentence' and 'entities' keys.
    'entities' is expected to be a list of dicts with 'text', 'label', 'start', 'end'.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f"✅ Loaded {len(data)} items from {file_path}")

        processed_data = []

        for item in data:
            # Extract sentence text
            sentence = item.get('sentence', '')

            # Clean up the "text': '" prefix from the sentence if it exists
            # Based on your snippet, it seems to be 'text': '
            if sentence.startswith("text': '"):
                sentence = sentence[len("text': '"):]
            if sentence.endswith("'"): # Remove trailing quote if present
                sentence = sentence[:-1]

            # Directly access the 'entities' list
            raw_entities = item.get('entities', [])
            
            entities = []
            for ent in raw_entities:
                # Map 'start' to 'start_char' and 'end' to 'end_char' for consistency
                # with your previous function's expected output format.
                if all(k in ent for k in ['text', 'label', 'start', 'end']):
                    entities.append({
                        'text': ent['text'],
                        'label': ent['label'],
                        'start_char': ent['start'],
                        'end_char': ent['end']
                    })

            processed_data.append({
                'sentence': sentence,
                'entities': entities,
                'original_data': item # Keep original data for reference if needed
            })

        # Print statistics
        total_entities = sum(len(item['entities']) for item in processed_data)
        sentences_with_entities = sum(1 for item in processed_data if item['entities'])

        print(f"\n📊 Processing Statistics:")
        print(f"  - Total sentences: {len(processed_data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        print(f"  - Average entities per sentence: {total_entities/len(processed_data):.2f}")

        return processed_data

    except Exception as e:
        logger.error(f"Error loading data: {e}")
        traceback.print_exc()
        return []

# --- Usage Example ---
# Assuming your file is named 'output_entities2.json' and is directly in the format you showed.
# Replace with your actual file path if different.
file_path = "output_entities2.json" 

input_data = load_data_from_custom_format(file_path)

# Verify the fix worked
if input_data:
    print("\n🔍 First item check:")
    first_item = input_data[0]
    print(f"Sentence: {first_item['sentence'][:100]}...") # Print first 100 chars
    print(f"Entities found: {len(first_item['entities'])}")
    if first_item['entities']:
        for ent in first_item['entities']:
            print(f"  - {ent['text']} ({ent['label']}) [Chars: {ent.get('start_char')}-{ent.get('end_char')}]")
    else:
        print("  No entities found for the first item after processing.")

✅ Loaded 67 items from output_entities2.json

📊 Processing Statistics:
  - Total sentences: 67
  - Sentences with entities: 67
  - Total entities extracted: 334
  - Average entities per sentence: 4.99

🔍 First item check:
Sentence: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often con...
Entities found: 2
  - the 1970s and 1980s (DATE) [Chars: 30-49]
  - loss (CHANGE) [Chars: 64-68]


In [63]:
def load_mistral_model(model_id, use_quantization=True):
    """Load Mistral model and tokenizer"""
    print(f"🔄 Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Configure quantization
        quantization_config = None
        if use_quantization and device == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if device == "cuda" else None,
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        
        print(f"✅ Model loaded successfully on {device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

# Load the model
model, tokenizer = load_mistral_model(CONFIG["model_id"], CONFIG["use_quantization"])

🔄 Loading model: mistralai/Mistral-7B-Instruct-v0.2
🔧 Using 4-bit quantization


2025-07-09 16:31:19,545 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Model loaded successfully on cuda


In [74]:
def build_lulc_extraction_prompt(sentence, entities):
    """Build prompt for LULC relation extraction"""
    
    # Format entities list
    entity_list = "\n".join([
        f"- {ent['text']} | {ent['label']}"
        for ent in entities
    ])
    
    prompt = f"""You are an expert in Land Use Land Cover (LULC) analysis. Extract ONLY the relations between entities from the given sentence.

SENTENCE: "{sentence}"

ENTITIES FOUND:
{entity_list}

**Relationship Types - BE VERY THOUGHTFUL:**

**CHANGE_TO**: Indicates a direct transformation from one LULC type to another.
- ✅ CORRECT: forest --CHANGE_TO-- cropland (trees cut, land converted to farming)
- ✅ CORRECT: agricultural land --CHANGE_TO-- urban area (farmland developed into city)
- ✅ CORRECT: grassland --CHANGE_TO-- built-up area (grass removed, buildings constructed)
- ❌ WRONG: built-up area --CHANGE_TO-- built-up area (same type, just quantity change)
- ❌ WRONG: forest --CHANGE_TO-- forest (same type, just area change)

**INCREASES_BY/DECREASES_BY**: For quantitative changes within same LULC type
- ✅ CORRECT: built-up area --INCREASES_BY-- 12.77% (more built-up area, not transformation)
- ✅ CORRECT: forest --DECREASES_BY-- 25% (less forest area, not transformation)

**Other Relations:**
- CAUSES: Process entity causes a change (deforestation --CAUSES-- forest loss)
- LOCATED_IN: Spatial relationships (forest --LOCATED_IN-- Brazil)
- OCCURS_DURING: Temporal relationships (change --OCCURS_DURING-- 2018)
- MEASURES: Quantitative relationships (12.77% --MEASURES-- increase)
- AFFECTS: Impact relationships (urbanization --AFFECTS-- forest)
- FROM_TO: Value changes (52.88% --FROM_TO-- 65.5%)
- ENABLES: One process enables another (deforestation --ENABLES-- urbanization)

**Relationship Types - BE VERY THOUGHTFUL:**
...
- OCCURS_DURING: Temporal relationships where a **change, process** takes place or is observed within a specific time period. (e.g., change --OCCURS_DURING-- 2018, urbanization --OCCURS_DURING-- decade)
❌ WRONG: simulation results --OCCURS_DURING-- study period (results don't 'occur' in time, they are 'from' or 'valid for' a period)

**CRITICAL THINKING RULES:**
1. **Ask yourself**: Is this ACTUALLY a transformation between different land types?
2. **Think about the process**: What physical change happened to the land?
3. **Consider causality**: What caused what? Don't create meaningless loops
4. **Be precise with measurements**: Percentages usually MEASURE changes, not cause them
5. **Temporal logic**: Changes happen DURING time periods, not TO time periods
6. **Spatial logic**: Things are LOCATED_IN places, places don't transform to places

INSTRUCTIONS:
1. Extract ONLY relations that are explicitly stated or directly implied in the sentence
2. Use ONLY the entities provided above
3. Each relation must include the entity label in the format: entity_text:ENTITY_LABEL
4. Each relation must follow the format: source_entity:SOURCE_LABEL --RELATIONSHIP-- target_entity:TARGET_LABEL
5. Include confidence level (HIGH/MEDIUM/LOW) for each relation
6. Do not create relations between entities of the same type using TRANSFORMS_TO

OUTPUT FORMAT:
RELATIONS:
- source_entity:SOURCE_LABEL --RELATIONSHIP-- target_entity:TARGET_LABEL | CONF: confidence_level

Example:
- forest:LULC_TYPE --CHANGE_TO-- cropland:LULC_TYPE | CONF: HIGH
- urbanization:PROCESS --CAUSES-- forest loss:CHANGE | CONF: MEDIUM
- 12.77%:PERCENTAGE --MEASURES-- increase:CHANGE | CONF: HIGH

Now extract relations from the given sentence:"""
    
    return prompt

# Test prompt building
if input_data and input_data[0]['entities']:
    test_prompt = build_lulc_extraction_prompt(
        input_data[0]['sentence'], 
        input_data[0]['entities']
    )
    print("Sample prompt (first 500 chars):")
    print(test_prompt[:500] + "...")

Sample prompt (first 500 chars):
You are an expert in Land Use Land Cover (LULC) analysis. Extract ONLY the relations between entities from the given sentence.

SENTENCE: "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel were designated as degraded land (e.g"

**Relationship Types - BE VERY THOUGHTFUL:**

**CHANGE_TO**: Indicates a direct transformation from one LULC type to another.
- ✅ CORRECT: forest --CHANGE_TO-- cropland...


In [76]:
def generate_relations(sentence, entities, model, tokenizer):
    """Generate relations using the model"""
    
    # Build prompt
    prompt = build_lulc_extraction_prompt(sentence, entities)
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3000,
        padding=True
    )
    
    # Move to device
    if device == "cuda":
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    
    # Decode response
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

# Test generation with first sentence
if input_data and input_data[0]['entities']:
    test_response = generate_relations(
        input_data[0]['sentence'],
        input_data[0]['entities'],
        model,
        tokenizer
    )
    print("Model response:")
    print(test_response)

/home/raham/venv/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Model response:


SENTENCE: "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel were designated as degraded land (e.g"

RELATIONS:
- droughts:EVENT --OCCURS_DURING-- 1970s:TIME_PERIOD, 1980s:TIME_PERIOD | CONF: HIGH
- woody_vegetation_cover:LULC_TYPE --DECREASES_BY-- loss:CHANGE | CONF: HIGH
- loss:CHANGE --CAUSES-- degraded_land:LULC_TYPE | CONF: HIGH
- degraded_land:LULC_TYPE --LOCATED_IN-- Sahel:REGION | CONF: HIGH


In [66]:
# Check the actual entities and their labels
if input_data and input_data[0]['entities']:
    print("Entities in first example:")
    for ent in input_data[0]['entities']:
        print(f"  - {ent['text']} | {ent['label']}")

Entities in first example:
  - the 1970s and 1980s | DATE
  - loss | CHANGE


In [71]:
# Updated valid types based on your new format
VALID_ENTITY_TYPES = {
    'CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 
    'COORDINATES', 'SURFACE_UNIT', 'PROCESS', 'QUANTITY' # Added new types from your example
}

VALID_RELATION_TYPES = {
    'CHANGE_TO', 'INCREASES_BY', 'DECREASES_BY', 'CAUSES', 
    'LOCATED_IN', 'OCCURS_DURING',  'AFFECTS', 
    'FROM_TO', 'ENABLES'  # Added new relation type
}

def parse_relations_from_response(response, sentence, valid_entities, log_all_relations=True):
    """Parse relations from model response and optionally log all generated relations"""
    import re
    import logging
    
    relations = []
    valid_entity_names = [e['text'].strip() for e in valid_entities]
    
    lines = response.strip().split('\n')
    in_relations_section = False
    
    # Track what entity types we're seeing for debugging
    seen_entity_types = set()
    skipped_relations = 0
    skipped_entity_types = 0
    
    for line in lines:
        line = line.strip()
        
        # Check if we're in the relations section
        if 'RELATIONS:' in line.upper():
            in_relations_section = True
            continue
            
        # Skip empty lines or lines that don't contain relations
        if not in_relations_section or not line or not ('--' in line):
            continue
            
        # Parse relations - Updated regex patterns
        patterns = [
            # Pattern for entity:type format with --> arrow
            r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)-->\s*(.+?):([A-Z_]+)\s*\|\s*CONF:\s*(\w+)',
            # Pattern for entity:type format with -- (no arrow)
            r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)--\s*(.+?):([A-Z_]+)\s*\|\s*CONF:\s*(\w+)',
            # Pattern with lowercase confidence
            r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)--\s*(.+?):([A-Z_]+)\s*\|\s*confidence:\s*(\w+)',
            # Pattern without confidence
            r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)-->\s*(.+?):([A-Z_]+)',
            r'^-?\s*(.+?):([A-Z_]+)\s*--([A-Z_]+)--\s*(.+?):([A-Z_]+)',
        ]
        
        parsed_relation = None
        
        for pattern in patterns:
            match = re.match(pattern, line)
            if match:
                groups = match.groups()
                
                # Extract components
                source = groups[0].strip()
                source_type = groups[1].strip().upper()  # Ensure uppercase
                relationship = groups[2].strip().upper()  # Ensure uppercase
                target = groups[3].strip()
                target_type = groups[4].strip().upper()  # Ensure uppercase
                confidence = groups[5].strip().upper() if len(groups) >= 6 else "MEDIUM"
                
                # Track entity types we're seeing
                seen_entity_types.add(source_type)
                seen_entity_types.add(target_type)
                
                parsed_relation = {
                    'source': source,
                    'source_type': source_type,
                    'relationship': relationship,
                    'target': target,
                    'target_type': target_type,
                    'confidence': confidence,
                    'original_line': line
                }
                
                if log_all_relations:
                    logging.debug(f"🔍 Parsed: {source}:{source_type} --{relationship}--> {target}:{target_type}")
                break
        
        # If no pattern matched, try fallback patterns for old format
        if not parsed_relation:
            fallback_patterns = [
                r'^-?\s*(.+?)\s*--([A-Z_]+)-->\s*(.+?)\s*\|\s*CONF:\s*(\w+)',
                r'^-?\s*(.+?)\s*--([A-Z_]+)--\s*(.+?)\s*\|\s*CONF:\s*(\w+)',
                r'^-?\s*(.+?)\s*--([A-Z_]+)--\s*(.+?)$',
            ]
            
            for pattern in fallback_patterns:
                match = re.match(pattern, line)
                if match:
                    groups = match.groups()
                    parsed_relation = {
                        'source': groups[0].strip(),
                        'source_type': 'UNKNOWN',
                        'relationship': groups[1].strip().upper(),
                        'target': groups[2].strip(),
                        'target_type': 'UNKNOWN',
                        'confidence': groups[3].strip().upper() if len(groups) >= 4 else "MEDIUM",
                        'original_line': line
                    }
                    if log_all_relations:
                        logging.debug(f"🔍 Fallback parsed: {parsed_relation}")
                    break
        
        # Process the parsed relation
        if parsed_relation:
            source = parsed_relation['source']
            source_type = parsed_relation['source_type']
            relationship = parsed_relation['relationship']
            target = parsed_relation['target']
            target_type = parsed_relation['target_type']
            confidence = parsed_relation['confidence']
            
            # FILTER 1: Skip if relationship type is invalid
            if relationship not in VALID_RELATION_TYPES:
                if log_all_relations:
                    logging.warning(f"❌ Skipping invalid relationship: {relationship}")
                skipped_relations += 1
                continue
            
            # FILTER 2: Skip if source entity type is invalid (except UNKNOWN)
            if source_type  not in VALID_ENTITY_TYPES:
                if log_all_relations:
                    logging.warning(f"❌ Skipping invalid source entity type: {source_type}")
                skipped_entity_types += 1
                continue
            
            # FILTER 3: Skip if target entity type is invalid (except UNKNOWN)
            if target_type not in VALID_ENTITY_TYPES:
                if log_all_relations:
                    logging.warning(f"❌ Skipping invalid target entity type: {target_type}")
                skipped_entity_types += 1
                continue
            
            # Handle placeholder targets
           
            
            # Clean up source and target to match valid entities
            source_cleaned = source
            target_cleaned = target
            
            # Try to match with valid entities (for backward compatibility)
            for valid_entity in valid_entity_names:
                if valid_entity.lower() in source.lower():
                    source_cleaned = valid_entity
                    break
            
            for valid_entity in valid_entity_names:
                if valid_entity.lower() in target.lower() and target not in ['x', 'y', 'z']:
                    target_cleaned = valid_entity
                    break
            
            # Additional validation checks
            if relationship == "CHANGE_TO" and source_cleaned.lower() == target_cleaned.lower():
                if log_all_relations:
                    logging.warning(f"❌ Skipping invalid CHANGE_TO: source and target are the same")
                continue
            
            if not source_cleaned.strip() or not target_cleaned.strip():
                if log_all_relations:
                    logging.warning(f"❌ Skipping relation with empty entities")
                continue
            
            # Add valid relation
            relations.append({
                'source': source_cleaned,
                'source_type': source_type,
                'relationship': relationship,
                'target': target_cleaned,
                'target_type': target_type,
                'confidence': confidence
            })
            
            if log_all_relations:
                logging.info(f"✅ ACCEPTED: {source_cleaned}:{source_type} --{relationship}--> {target_cleaned}:{target_type}")
        
        else:
            # Log unparseable lines
            if log_all_relations:
                logging.warning(f"⚠️ Could not parse line: '{line}'")
    
    # Debug: Show all entity types we encountered
    if log_all_relations and seen_entity_types:
        logging.info(f"🏷️ Entity types found in response: {sorted(seen_entity_types)}")
        invalid_types = seen_entity_types - VALID_ENTITY_TYPES - {'UNKNOWN'}
        if invalid_types:
            logging.warning(f"⚠️ Invalid entity types detected: {sorted(invalid_types)}")
        else:
            logging.info(f"✅ All entity types are valid")
    
    if log_all_relations:
        logging.info(f"📊 Summary: {len(relations)} valid relations extracted")
        logging.info(f"📊 Skipped {skipped_relations} relations due to invalid relationship types")
        logging.info(f"📊 Skipped {skipped_entity_types} relations due to invalid entity types")
    
    return relations

In [72]:
def process_all_sentences(input_data, model, tokenizer, max_sentences=None):
    """Process all sentences to extract LULC relations"""
    
    # Limit processing if specified
    if max_sentences:
        input_data = input_data[:max_sentences]
        print(f"🔄 Processing {max_sentences} sentences (limited for testing)")
    else:
        print(f"🔄 Processing all {len(input_data)} sentences")
    
    all_results = []
    successful_extractions = 0
    failed_extractions = 0
    
    # Track entity type statistics
    entity_type_stats = {}
    relation_type_stats = {}
    
    # Process each sentence
    for idx, item in enumerate(tqdm(input_data, desc="Extracting relations")):
        sentence = item['sentence']
        entities = item['entities']
        
        # Skip sentences without entities
        if not entities:
            logger.info(f"Skipping sentence {idx}: No entities found")
            continue
        
        try:
            # Generate relations
            response = generate_relations(sentence, entities, model, tokenizer)
            
            # Parse relations from response
            relations = parse_relations_from_response(response, sentence, entities)
            
            # Update statistics
            for relation in relations:
                # Track entity types
                source_type = relation.get('source_type', 'UNKNOWN')
                target_type = relation.get('target_type', 'UNKNOWN')
                relation_type = relation.get('relationship', 'UNKNOWN')
                
                entity_type_stats[source_type] = entity_type_stats.get(source_type, 0) + 1
                entity_type_stats[target_type] = entity_type_stats.get(target_type, 0) + 1
                relation_type_stats[relation_type] = relation_type_stats.get(relation_type, 0) + 1
            
            # Store results
            result = {
                'sentence_id': idx,
                'sentence': sentence,
                'entities': entities,
                'model_response': response,
                'extracted_relations': relations,
                'num_relations': len(relations),
                'processing_timestamp': datetime.now().isoformat()
            }
            
            all_results.append(result)
            
            if relations:
                successful_extractions += 1
                logger.info(f"✅ Sentence {idx}: Found {len(relations)} relations")
                # Log detailed relation info
                for rel in relations:
                    logger.debug(f"   Relation: {rel['source']}:{rel.get('source_type', 'UNKNOWN')} "
                               f"--{rel['relationship']}--> {rel['target']}:{rel.get('target_type', 'UNKNOWN')} "
                               f"| {rel['confidence']}")
            else:
                failed_extractions += 1
                logger.warning(f"⚠️ Sentence {idx}: No valid relations extracted")
                
        except Exception as e:
            logger.error(f"❌ Error processing sentence {idx}: {e}")
            failed_extractions += 1
            
            # Store error result
            error_result = {
                'sentence_id': idx,
                'sentence': sentence,
                'entities': entities,
                'model_response': f"ERROR: {str(e)}",
                'extracted_relations': [],
                'num_relations': 0,
                'processing_timestamp': datetime.now().isoformat(),
                'error': str(e)
            }
            all_results.append(error_result)
    
    # Print summary statistics
    print(f"\n📊 Processing Complete!")
    print(f"  - Total sentences processed: {len(input_data)}")
    print(f"  - Successful extractions: {successful_extractions}")
    print(f"  - Failed extractions: {failed_extractions}")
    print(f"  - Success rate: {(successful_extractions/(successful_extractions+failed_extractions)*100):.1f}%")
    
    # Calculate relation statistics
    total_relations = sum(len(result['extracted_relations']) for result in all_results)
    sentences_with_relations = sum(1 for result in all_results if result['extracted_relations'])
    
    print(f"  - Total relations extracted: {total_relations}")
    print(f"  - Sentences with relations: {sentences_with_relations}")
    if sentences_with_relations > 0:
        print(f"  - Average relations per successful sentence: {total_relations/sentences_with_relations:.2f}")
    
    # Print entity type statistics
    if entity_type_stats:
        print(f"\n📈 Entity Type Distribution:")
        sorted_entity_types = sorted(entity_type_stats.items(), key=lambda x: x[1], reverse=True)
        for entity_type, count in sorted_entity_types:
            print(f"  - {entity_type}: {count}")
    
    # Print relation type statistics
    if relation_type_stats:
        print(f"\n🔗 Relation Type Distribution:")
        sorted_relation_types = sorted(relation_type_stats.items(), key=lambda x: x[1], reverse=True)
        for relation_type, count in sorted_relation_types:
            print(f"  - {relation_type}: {count}")
    
    return all_results

# Process all sentences (or limited number for testing)
print("🚀 Starting relation extraction...")
results = process_all_sentences(
    input_data, 
    model, 
    tokenizer, 
    max_sentences=CONFIG["max_sentences"]  # Set to None to process all
)

# Save results to JSON file
output_file = Path(CONFIG["output_dir"]) / f"lulc_relations_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"💾 Results saved to: {output_file}")

# Display sample results with enhanced formatting
print(f"\n🔍 Sample Results (first 3 successful extractions):")
successful_results = [r for r in results if r['extracted_relations']]

for i, result in enumerate(successful_results[:3]):
    print(f"\n--- Sample {i+1} ---")
    print(f"Sentence: {result['sentence'][:100]}...")
    print(f"Relations found: {result['num_relations']}")
    
    for rel in result['extracted_relations']:
        # Enhanced display with entity types
        source_display = f"{rel['source']}:{rel.get('source_type', 'UNKNOWN')}"
        target_display = f"{rel['target']}:{rel.get('target_type', 'UNKNOWN')}"
        
        print(f"  ➤ {source_display} --{rel['relationship']}--> {target_display} | CONF: {rel['confidence']}")

# Additional analysis: Show relation patterns
print(f"\n🔍 Relation Patterns Analysis:")
if successful_results:
    # Analyze most common relation patterns
    pattern_counts = {}
    for result in successful_results:
        for rel in result['extracted_relations']:
            source_type = rel.get('source_type', 'UNKNOWN')
            relation_type = rel.get('relationship', 'UNKNOWN')
            target_type = rel.get('target_type', 'UNKNOWN')
            
            pattern = f"{source_type} --{relation_type}--> {target_type}"
            pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1
    
    # Show top 10 patterns
    top_patterns = sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    print("Top 10 Relation Patterns:")
    for pattern, count in top_patterns:
        print(f"  - {pattern}: {count} occurrences")

# Validation check for entity types
print(f"\n✅ Validation Check:")
invalid_entity_types = set()
invalid_relation_types = set()

for result in results:
    for rel in result.get('extracted_relations', []):
        source_type = rel.get('source_type', 'UNKNOWN')
        target_type = rel.get('target_type', 'UNKNOWN')
        relation_type = rel.get('relationship', 'UNKNOWN')
        
        if source_type not in VALID_ENTITY_TYPES and source_type != 'UNKNOWN':
            invalid_entity_types.add(source_type)
        if target_type not in VALID_ENTITY_TYPES and target_type != 'UNKNOWN':
            invalid_entity_types.add(target_type)
        if relation_type not in VALID_RELATION_TYPES:
            invalid_relation_types.add(relation_type)

if invalid_entity_types:
    print(f"⚠️ Invalid entity types found: {invalid_entity_types}")
else:
    print("✅ All entity types are valid")

if invalid_relation_types:
    print(f"⚠️ Invalid relation types found: {invalid_relation_types}")
else:
    print("✅ All relation types are valid")

🚀 Starting relation extraction...
🔄 Processing 20 sentences (limited for testing)


Extracting relations:   0%|          | 0/20 [00:00<?, ?it/s]

/home/raham/venv/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
2025-07-09 16:50:48,788 - INFO - ✅ ACCEPTED: droughts:DATE --CAUSES--> loss:CHANGE
2025-07-09 16:50:48,792 - INFO - ✅ ACCEPTED: loss:CHANGE --OCCURS_DURING--> 1970s and 1980s:DATE
2025-07-09 16:50:48,793 - WARNING - ❌ Skipping invalid target entity type: LULC_TYPE
2025-07-09 16:50:48,794 - WARNING - ❌ Skipping invalid source entity type: LULC_TYPE
2025-07-09 16:50:48,795 - WARNING - ❌ Skipping invalid source entity type: ADJECTIVE
2025-07-09 16:50:48,796 - WARNING - ❌ Skipping invalid relationship: DESIGNATED_AS
2025-07-09 16:50:48,797 - INFO - 🏷️ Entity types found in response: ['ADJECTIVE', 'CHANGE', 'DATE', 'LULC_TYPE', 'QUANTITY']
2025-07-09 16:50:48,797 - WARNING - ⚠️ Inval


📊 Processing Complete!
  - Total sentences processed: 20
  - Successful extractions: 9
  - Failed extractions: 11
  - Success rate: 45.0%
  - Total relations extracted: 19
  - Sentences with relations: 9
  - Average relations per successful sentence: 2.11

📈 Entity Type Distribution:
  - CHANGE: 15
  - PROCESS: 12
  - DATE: 5
  - LULC: 2
  - LOC: 2
  - PERCENT: 1
  - QUANTITY: 1

🔗 Relation Type Distribution:
  - CAUSES: 8
  - OCCURS_DURING: 4
  - AFFECTS: 3
  - INCREASES_BY: 2
  - LOCATED_IN: 1
  - ENABLES: 1
💾 Results saved to: lulc_extraction_output/lulc_relations_20250709_165543.json

🔍 Sample Results (first 3 successful extractions):

--- Sample 1 ---
Sentence: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often con...
Relations found: 2
  ➤ droughts:DATE --CAUSES--> loss:CHANGE | CONF: HIGH
  ➤ loss:CHANGE --OCCURS_DURING--> 1970s and 1980s:DATE | CONF: HIGH

--- Sample 2 ---
Sentence: ).',, 'p': 'ref':, ' text': 'From 1996 to 2017, t

In [73]:
def save_extraction_results(results, output_dir):
    """Save results in multiple formats"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Compute basic statistics
    stats = {
        'total_sentences_processed': len(results),
        'sentences_with_relations': sum(1 for r in results if r.get('extracted_relations')),
        'total_relations_extracted': sum(len(r.get('extracted_relations', [])) for r in results),
        'relation_types': {},
        'entity_types': {}
    }
    
    # Count relation types and entity types
    for result in results:
        for relation in result.get('extracted_relations', []):
            relation_type = relation.get('relationship', 'UNKNOWN')
            source_type = relation.get('source_type', 'UNKNOWN')
            target_type = relation.get('target_type', 'UNKNOWN')
            
            stats['relation_types'][relation_type] = stats['relation_types'].get(relation_type, 0) + 1
            stats['entity_types'][source_type] = stats['entity_types'].get(source_type, 0) + 1
            stats['entity_types'][target_type] = stats['entity_types'].get(target_type, 0) + 1
    
    # 1. Save detailed results
    detailed_output = {
        'metadata': {
            'extraction_date': datetime.now().isoformat(),
            'model': CONFIG['model_id'],
            'statistics': stats
        },
        'results': results
    }
    
    detailed_path = Path(output_dir) / f"relations_extracted_{timestamp}.json"
    with open(detailed_path, 'w', encoding='utf-8') as f:
        json.dump(detailed_output, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Detailed results saved to: {detailed_path}")
    
    # 2. Save simplified CSV format with source and target labels
    csv_data = []
    for result in results:
        sentence_id = result.get('sentence_id', 'N/A')
        sentence = result.get('sentence', 'N/A')
        for relation in result.get('extracted_relations', []):
            csv_data.append({
                'sentence_id': sentence_id,
                'sentence': sentence,
                'source': relation['source'],
                'source_label': relation.get('source_type', 'UNKNOWN'),
                'relationship': relation['relationship'],
                'target': relation['target'],
                'target_label': relation.get('target_type', 'UNKNOWN'),
                'confidence': relation.get('confidence', 'MEDIUM')
            })
    
    if csv_data:
        df = pd.DataFrame(csv_data)
        csv_path = Path(output_dir) / f"relations_table_{timestamp}.csv"
        df.to_csv(csv_path, index=False)
        print(f"✅ CSV table saved to: {csv_path}")
        
        # Print sample of CSV data
        print(f"\n📋 CSV Preview (first 5 rows):")
        print(df.head().to_string(index=False))
    
    # 3. Save statistics with entity type breakdown
    stats_path = Path(output_dir) / f"statistics_{timestamp}.json"
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    
    print(f"✅ Statistics saved to: {stats_path}")
    
    # 4. Create a summary report
    summary_path = Path(output_dir) / f"extraction_summary_{timestamp}.txt"
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write(f"LULC Relation Extraction Summary\n")
        f.write(f"Generated: {datetime.now().isoformat()}\n")
        f.write(f"Model: {CONFIG['model_id']}\n")
        f.write("=" * 50 + "\n\n")
        
        f.write(f"OVERALL STATISTICS:\n")
        f.write(f"- Total sentences processed: {stats['total_sentences_processed']}\n")
        f.write(f"- Sentences with relations: {stats['sentences_with_relations']}\n")
        f.write(f"- Total relations extracted: {stats['total_relations_extracted']}\n")
        f.write(f"- Success rate: {(stats['sentences_with_relations']/stats['total_sentences_processed']*100):.1f}%\n\n")
        
        f.write(f"RELATION TYPE DISTRIBUTION:\n")
        sorted_relations = sorted(stats['relation_types'].items(), key=lambda x: x[1], reverse=True)
        for rel_type, count in sorted_relations:
            f.write(f"- {rel_type}: {count}\n")
        
        f.write(f"\nENTITY TYPE DISTRIBUTION:\n")
        sorted_entities = sorted(stats['entity_types'].items(), key=lambda x: x[1], reverse=True)
        for ent_type, count in sorted_entities:
            f.write(f"- {ent_type}: {count}\n")
    
    print(f"✅ Summary report saved to: {summary_path}")
    
    # Print detailed statistics
    print(f"\n📊 Extraction Statistics:")
    print(f"  - Total sentences: {stats['total_sentences_processed']}")
    print(f"  - Sentences with relations: {stats['sentences_with_relations']}")
    print(f"  - Total relations: {stats['total_relations_extracted']}")
    print(f"  - Success rate: {(stats['sentences_with_relations']/stats['total_sentences_processed']*100):.1f}%")
    
    if stats['relation_types']:
        print(f"\n🔗 Top Relation Types:")
        sorted_relations = sorted(stats['relation_types'].items(), key=lambda x: x[1], reverse=True)[:5]
        for rel_type, count in sorted_relations:
            print(f"  - {rel_type}: {count}")
    
    if stats['entity_types']:
        print(f"\n🏷️ Top Entity Types:")
        sorted_entities = sorted(stats['entity_types'].items(), key=lambda x: x[1], reverse=True)[:5]
        for ent_type, count in sorted_entities:
            print(f"  - {ent_type}: {count}")
    
    return detailed_path, csv_path if csv_data else None, stats_path, summary_path

# Save all results
print("💾 Saving extraction results...")
paths = save_extraction_results(results, CONFIG["output_dir"])

print(f"\n📁 All files saved to: {CONFIG['output_dir']}")
print("Files generated:")
for i, path in enumerate(paths):
    if path:
        file_types = ["Detailed JSON", "CSV Table", "Statistics", "Summary Report"]
        print(f"  {i+1}. {file_types[i]}: {Path(path).name}")

💾 Saving extraction results...
✅ Detailed results saved to: lulc_extraction_output/relations_extracted_20250709_165543.json
✅ CSV table saved to: lulc_extraction_output/relations_table_20250709_165543.csv

📋 CSV Preview (first 5 rows):
 sentence_id                                                                                                                                                                                                                           sentence             source source_label  relationship                         target target_label confidence
           0                                     After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel were designated as degraded land (e.g           droughts         DATE        CAUSES                           loss       CHANGE       HIGH
           0                                     After the droughts in the 1970s a

In [70]:
def create_label_studio_output(results, original_data, output_dir):
    """Create Label Studio compatible output with relations - FIXED VERSION"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    label_studio_tasks = []
    
    for idx, (result, orig_item) in enumerate(zip(results, original_data)):
        # Start with predictions structure
        predictions = []
        entity_id_map = {}
        entity_counter = 0
        
        # Get the original text
        text = orig_item['original_data']['data']['text']
        
        # Step 1: Add entities from original annotations
        if 'annotations' in orig_item['original_data'] and orig_item['original_data']['annotations']:
            for annotation in orig_item['original_data']['annotations']:
                if 'result' in annotation:
                    for res in annotation['result']:
                        if res['type'] == 'labels':
                            entity_id = f"ent_{idx}_{entity_counter}"
                            entity_counter += 1
                            entity_id_map[res['value']['text']] = entity_id
                            
                            # Add entity prediction
                            entity_pred = res.copy()
                            entity_pred['id'] = entity_id
                            predictions.append(entity_pred)
        
        # Step 2: Check for entities from model extraction that might not be in annotations
        for entity in result.get('entities', []):
            if entity['text'] not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[entity['text']] = entity_id
                
                # Create new entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': entity['start_char'] if entity['start_char'] >= 0 else 0,
                        'end': entity['end_char'] if entity['end_char'] >= 0 else len(entity['text']),
                        'text': entity['text'],
                        'labels': [entity['label']]
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
        
        # Step 3: Process relations and create missing entities
        for rel_idx, relation in enumerate(result['relations']):
            source = relation['source']
            target = relation['target']
            
            # If source entity not found, create it
            if source not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[source] = entity_id
                
                # Try to find position in text
                source_start = text.lower().find(source.lower())
                source_end = source_start + len(source) if source_start != -1 else -1
                
                # Create entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': source_start if source_start >= 0 else 0,
                        'end': source_end if source_end >= 0 else len(source),
                        'text': source,
                        'labels': ['ENTITY']  # Default label for missing entities
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
                logger.info(f"Created new entity for relation source: {source}")
            
            # If target entity not found, create it
            if target not in entity_id_map:
                entity_id = f"ent_{idx}_{entity_counter}"
                entity_counter += 1
                entity_id_map[target] = entity_id
                
                # Try to find position in text
                target_start = text.lower().find(target.lower())
                target_end = target_start + len(target) if target_start != -1 else -1
                
                # Create entity prediction
                entity_pred = {
                    'id': entity_id,
                    'type': 'labels',
                    'value': {
                        'start': target_start if target_start >= 0 else 0,
                        'end': target_end if target_end >= 0 else len(target),
                        'text': target,
                        'labels': ['ENTITY']  # Default label for missing entities
                    },
                    'from_name': 'label',
                    'to_name': 'text'
                }
                predictions.append(entity_pred)
                logger.info(f"Created new entity for relation target: {target}")
            
            # Now add the relation
            relation_pred = {
                'from_id': entity_id_map[source],
                'to_id': entity_id_map[target],
                'type': 'relation',
                'labels': [relation['relationship']],
                'direction': 'right',
                'meta': {
                    'confidence': relation['confidence']
                }
            }
            predictions.append(relation_pred)
        
        # Create Label Studio task
        task = {
            'data': orig_item['original_data']['data'],
            'predictions': [{
                'model_version': f'lulc_relations_{timestamp}',
                'result': predictions
            }]
        }
        
        # Keep original annotations if present
        if 'annotations' in orig_item['original_data']:
            task['annotations'] = orig_item['original_data']['annotations']
        
        label_studio_tasks.append(task)
    
    # Save Label Studio format
    ls_path = Path(output_dir) / f"label_studio_with_relations_{timestamp}.json"
    with open(ls_path, 'w', encoding='utf-8') as f:
        json.dump(label_studio_tasks, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Label Studio format saved to: {ls_path}")
    print(f"📊 Created {len(label_studio_tasks)} tasks with entities and relations")
    
    # Count statistics
    total_predictions = sum(len(task['predictions'][0]['result']) for task in label_studio_tasks)
    print(f"📊 Total predictions (entities + relations): {total_predictions}")
    
    # Create Label Studio config XML with all entity types
    config_xml = """<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
    <Label value="LULC" background="#FF6B6B"/>
    <Label value="DATE" background="#4ECDC4"/>
    <Label value="LOCATION" background="#45B7D1"/>
    <Label value="PERCENT" background="#96CEB4"/>
    <Label value="PROCESS" background="#FECA57"/>
    <Label value="QUANTITY" background="#FF9FF3"/>
    <Label value="CHANGE" background="#A55EEA"/>
    <Label value="ENTITY" background="#B4B4B4"/>
  </Labels>
  <Relations name="relation" toName="label">
    <Relation value="TRANSFORMS_TO" background="#FF6B6B"/>
    <Relation value="INCREASES_BY" background="#4ECDC4"/>
    <Relation value="DECREASES_BY" background="#45B7D1"/>
    <Relation value="CAUSES" background="#96CEB4"/>
    <Relation value="LOCATED_IN" background="#FECA57"/>
    <Relation value="OCCURS_DURING" background="#FF9FF3"/>
    <Relation value="MEASURES" background="#A55EEA"/>
    <Relation value="AFFECTS" background="#54A0FF"/>
    <Relation value="FROM_TO" background="#5F27CD"/>
    <Relation value="ENABLES" background="#00D2D3"/>
  </Relations>
</View>"""
    
    config_path = Path(output_dir) / "label_studio_config.xml"
    with open(config_path, 'w') as f:
        f.write(config_xml)
    
    print(f"✅ Label Studio config saved to: {config_path}")
    
    return ls_path

# Create Label Studio output with fixed function
ls_output_path = create_label_studio_output(results, input_data, CONFIG["output_dir"])

KeyError: 'data'

In [ ]:
def create_minimal_test_file(output_dir):
    """Create a minimal test file to verify Label Studio import works"""
    
    # Minimal working example
    test_task = {
        "data": {
            "text": "The forest area decreased by 25% in Brazil during 2018."
        },
        "annotations": [{
            "id": "test_annotation_1",
            "completed_by": 1,
            "result": [
                {
                    "id": "entity_1",
                    "type": "labels",
                    "value": {
                        "start": 4,
                        "end": 15,
                        "text": "forest area",
                        "labels": ["LULC"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "id": "entity_2",
                    "type": "labels",
                    "value": {
                        "start": 29,
                        "end": 32,
                        "text": "25%",
                        "labels": ["PERCENT"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "id": "entity_3",
                    "type": "labels",
                    "value": {
                        "start": 36,
                        "end": 42,
                        "text": "Brazil",
                        "labels": ["LOCATION"]
                    },
                    "from_name": "label",
                    "to_name": "text"
                },
                {
                    "from_id": "entity_1",
                    "to_id": "entity_2",
                    "type": "relation",
                    "labels": ["DECREASES_BY"],
                    "direction": "right"
                }
            ]
        }]
    }
    
    # Save minimal test
    test_path = Path(output_dir) / "minimal_test.json"
    with open(test_path, 'w') as f:
        json.dump([test_task], f, indent=2)
    
    print(f"✅ Minimal test file saved to: {test_path}")
    print("Try importing this file first!")
    
    return test_path

# Create minimal test
minimal_test_path = create_minimal_test_file(CONFIG["output_dir"])